# Module 10: Custom Pipeline Components


## 🔌 What is a Custom Pipeline Component?

As you learned in Module 3, spaCy's processing pipeline is just a series of functions that take a `Doc` object, modify it, and return it.

While spaCy comes with powerful built-in components (Tagger, Parser, NER), you will often want to inject your own custom logic directly into the pipeline so that it runs automatically every time you call `nlp(text)`. 

This is the core of extending spaCy for your specific domain!


<br><br>

---

<br><br>


## 🛠️ The `@Language.component` Decorator

The easiest way to write a custom component is to write a standard Python function that takes a `Doc` and returns a `Doc`, and then decorate it with `@Language.component`.

Let's write a component that counts the number of words in a document and prints it out automatically.


In [1]:
import spacy
from spacy.language import Language

# 1. Define the custom component
@Language.component("word_counter")
def custom_word_counter(doc):
    # Count words (excluding punctuation and spaces)
    word_count = len([t for t in doc if not t.is_punct and not t.is_space])
    print(f"[Word Counter] This document has {word_count} words.")
    
    # A component MUST return the doc!
    return doc

# 2. Create an empty English model
nlp = spacy.blank("en")

# 3. Add the component to the pipeline
nlp.add_pipe("word_counter")

# 4. Test it!
doc = nlp("Hello world! This is a custom spaCy pipeline.")
doc2 = nlp("Just one more test.")


[Word Counter] This document has 8 words.
[Word Counter] This document has 4 words.


<br><br>

---

<br><br>


## ⚙️ Configurable Components (`@Language.factory`)

What if you want to pass settings or arguments to your component? A simple `@Language.component` function cannot hold state or accept initialization arguments.

For that, we use the `@Language.factory` decorator. A factory is a function that *returns* a callable class (or function). This allows you to define a class with an `__init__` method to hold configuration!


In [2]:
from spacy.language import Language

# 1. Define a class that behaves like a component
class KeywordHighlighter:
    def __init__(self, nlp, name, keywords):
        self.keywords = [kw.lower() for kw in keywords]

    # The __call__ method is what spaCy runs on the Doc
    def __call__(self, doc):
        found = []
        for token in doc:
            if token.lower_ in self.keywords:
                found.append(token.text)
        
        if found:
            print(f"[Keyword Highlighter] Found keywords: {found}")
        return doc

# 2. Register the factory
# We define default arguments here
@Language.factory("keyword_highlighter", default_config={"keywords": ["important", "urgent"]})
def create_keyword_highlighter(nlp, name, keywords):
    return KeywordHighlighter(nlp, name, keywords)

# 3. Create a new pipeline and add the factory
nlp2 = spacy.blank("en")

# We can override the default keywords by passing config
nlp2.add_pipe("keyword_highlighter", config={"keywords": ["error", "warning", "critical"]})

doc = nlp2("This is a critical system error that needs attention.")


[Keyword Highlighter] Found keywords: ['critical', 'error']


<br><br>

---

<br><br>


## 🧭 Pipeline Ordering & Requirements

When you add a component, you can dictate exactly where it goes using `first=True`, `last=True`, `before="ner"`, or `after="tagger"`.

But what if your custom component *relies* on another component? For example, if you want to write a component that groups NOUNs, it will crash if the `tagger` hasn't run yet.

You can explicitly declare what your component requires using `assigns` and `requires` inside the decorator!


In [3]:
import spacy
from spacy.language import Language

# We explicitly state that this component requires POS tags to function
@Language.component("noun_printer", requires=["token.pos"])
def print_nouns(doc):
    nouns = [t.text for t in doc if t.pos_ == "NOUN"]
    print(f"Nouns found: {nouns}")
    return doc

nlp = spacy.load("en_core_web_sm")

# Because we declared 'requires', spaCy knows it's safest to put this 
# after the tagger, but we can also be explicit:
nlp.add_pipe("noun_printer", after="tagger")

doc = nlp("The developer wrote a great pipeline for the application.")


Nouns found: []


<br><br>

---

<br><br>


## 🧠 Practical Example: Rule-based Sentiment

Let's build a very simple, practical component: A dictionary-based sentiment analyzer.
*Note: In the next module (Module 11), we will learn how to actually save this sentiment score directly onto the `Doc` object using Extension Attributes! For now, we will just print it.*


In [4]:
@Language.component("simple_sentiment")
def simple_sentiment(doc):
    positive_words = {"good", "great", "excellent", "happy", "love"}
    negative_words = {"bad", "terrible", "awful", "sad", "hate"}
    
    score = 0
    for token in doc:
        if token.lemma_.lower() in positive_words:
            score += 1
        elif token.lemma_.lower() in negative_words:
            score -= 1
            
    if score > 0:
        print("Sentiment: Positive 🟢")
    elif score < 0:
        print("Sentiment: Negative 🔴")
    else:
        print("Sentiment: Neutral ⚪")
        
    return doc

nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("simple_sentiment", last=True)

doc1 = nlp("I love this product, it is absolutely great!")
doc2 = nlp("This is a terrible and bad experience.")
doc3 = nlp("I am going to the store.")


Sentiment: Positive 🟢
Sentiment: Negative 🔴
Sentiment: Neutral ⚪


<br><br>

---

<br><br>


## 🎉 Summary of Module 10

You now know how to extend spaCy beyond its defaults!
- You learned how to write simple stateless components using `@Language.component`.
- You learned how to write stateful, configurable components using Classes and `@Language.factory`.
- You learned how to enforce component ordering by declaring what a component `requires`.

However, right now our components are just *printing* things to the console. They aren't actually saving data back to the `Doc`! 

In **Module 11**, we will learn about **Extension Attributes**, which will allow us to attach our custom sentiment scores or keywords permanently to the `Doc`, `Token`, and `Span` objects!
